# Claude로 SQL 쿼리 만들기


이 노트북에서는 Claude를 사용해 자연어 질문으로부터 SQL 쿼리를 생성하는 방법을 살펴봅니다. 테스트 데이터베이스를 만들고, Claude에 스키마를 전달한 뒤, Claude가 사람의 말을 이해해 SQL 쿼리로 옮기는 과정을 확인합니다.

## 준비

먼저 필요한 라이브러리를 설치하고, API 키로 Anthropic 클라이언트를 설정합니다.

In [ ]:
# Install the necessary libraries
%pip install anthropic

In [2]:
# Import the required libraries
import sqlite3

from anthropic import Anthropic

# Set up the Claude API client
client = Anthropic()
MODEL_NAME = "claude-opus-4-1"

## 테스트 데이터베이스 만들기

SQLite로 테스트 데이터베이스를 만들고 샘플 데이터를 채워 넣습니다:

In [3]:
# Connect to the test database (or create it if it doesn't exist)
conn = sqlite3.connect("test_db.db")
cursor = conn.cursor()

# Create a sample table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER
    )
""")

# Insert sample data
sample_data = [
    (1, "John Doe", "Sales", 50000),
    (2, "Jane Smith", "Engineering", 75000),
    (3, "Mike Johnson", "Sales", 60000),
    (4, "Emily Brown", "Engineering", 80000),
    (5, "David Lee", "Marketing", 55000),
]
cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", sample_data)
conn.commit()

## Claude로 SQL 쿼리 생성하기

이제 자연어 질문을 Claude에 보내 생성된 SQL 쿼리를 돌려받는 함수를 정의합니다:

In [8]:
# Define a function to send a query to Claude and get the response
def ask_claude(query, schema):
    prompt = f"""Here is the schema for a database:

{schema}

Given this schema, can you output a SQL query to answer the following question? Only output the SQL query and nothing else.

Question: {query}
"""

    response = client.messages.create(
        model=MODEL_NAME, max_tokens=2048, messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

데이터베이스 스키마를 조회해 문자열 형태로 만듭니다:

In [5]:
# Get the database schema
schema = cursor.execute("PRAGMA table_info(employees)").fetchall()
schema_str = (
    "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
)
print(schema_str)

CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
department TEXT
salary INTEGER
)


이제 예시 자연어 질문을 만들어 Claude에 보내 봅니다:

In [9]:
# Example natural language question
question = "What are the names and salaries of employees in the Engineering department?"
# Send the question to Claude and get the SQL query
sql_query = ask_claude(question, schema_str)
print(sql_query)

SELECT name, salary
FROM EMPLOYEES
WHERE department = 'Engineering';


## 생성된 SQL 쿼리 실행하기

마지막으로 생성된 SQL 쿼리를 테스트 데이터베이스에서 실행하고 결과를 출력합니다:

In [10]:
# Execute the SQL query and print the results
results = cursor.execute(sql_query).fetchall()

for row in results:
    print(row)

('Jane Smith', 75000)
('Emily Brown', 80000)


작업이 끝나면 데이터베이스 연결을 닫는 것을 잊지 마세요:

In [11]:
# Close the database connection
conn.close()